# SOEN Disruption Prediction — PCA-8 Per-Sequence Binary Classification

Train a neuromorphic SOEN model on the PCA-8 + 100× decimated + flattop dataset.

**Hardware mapping**: The SOEN chip has **28 dendrite neurons**, each with **8 synaptic inputs**.
At each of 279 timesteps, all 28×8 = 224 input values are fed simultaneously.
The neurons update recurrently (28→28), then the next 224 inputs arrive.
After 279 steps, the first 24 neurons are read via SQUID and projected to 2 classes.

**Data reshape**: `(N, 7812, 8)` → `(N, 279, 224)` — 28 groups of 8 PCA features per step.

**Architecture**: `Linear(224) → SingleDendrite(28, rec) → SQUID(24) → Linear(2)`

**Trainable** (3-bit QAT): J_0_to_1 (224→28), J_1_to_1 (28→28 recurrent)
**Fixed**: J_1_to_2 (one-to-one, first 24 of 28, J=0.5), physics params
**Off-chip**: J_2_to_3 (24→2 linear, float, NOT quantized)

**Deliverable**: Quantized `.soen` checkpoint for hardware deployment.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

import numpy as np
import h5py
import yaml

# ── Find soen_toolkit src ─────────────────────────────────────────
_SOEN_SRC_CANDIDATES = [
    Path("/home/idies/workspace/Temporary/dpark1/scratch/soenhardware/soen-toolkit/src"),
    Path("/home/idies/workspace/Temporary/dpark1/scratch/SOEN/soenre2/src"),
    Path("/Users/davidpark/Documents/Cursor/soenhardware/soen-toolkit/src"),
]

SOEN_SRC = None
for _c in _SOEN_SRC_CANDIDATES:
    if (_c / "soen_toolkit" / "__init__.py").exists():
        SOEN_SRC = _c
        break
if SOEN_SRC is None:
    raise FileNotFoundError("soen_toolkit src not found")

TUTORIAL_DIR = SOEN_SRC / "soen_toolkit" / "tutorial_notebooks" / "time_to_event_tutorial"
print(f"soen_toolkit src: {SOEN_SRC}")

# ── Find a Python interpreter that can actually import soen_toolkit ──
# The Jupyter kernel may be Python 3.9 which can't handle `str | Path` syntax.
# Search for uv-managed or conda Python ≥3.10 that works.
_PYTHON_CANDIDATES = [
    # uv-managed venv in the soen-toolkit project
    SOEN_SRC.parent / ".venv" / "bin" / "python",
    SOEN_SRC.parent / ".venv" / "bin" / "python3",
    # conda envs
    Path("/home/idies/workspace/Temporary/dpark1/scratch/conda/conda_envs/soen/bin/python"),
    # system
    shutil.which("python3.11") or "",
    shutil.which("python3.10") or "",
    sys.executable,  # fallback to kernel python
]

SOEN_PYTHON = None
for _p in _PYTHON_CANDIDATES:
    _p = Path(str(_p))
    if not _p.exists():
        continue
    # Test if this python can import soen_toolkit
    _test = subprocess.run(
        [str(_p), "-c", "import soen_toolkit; print('OK')"],
        env={**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)},
        capture_output=True, text=True,
    )
    if _test.returncode == 0 and "OK" in _test.stdout:
        SOEN_PYTHON = _p
        break

if SOEN_PYTHON is None:
    raise RuntimeError(
        "No Python interpreter can import soen_toolkit. Tried:\n" +
        "\n".join(f"  {p}" for p in _PYTHON_CANDIDATES if Path(str(p)).exists())
    )

# Get Python version
_ver = subprocess.run([str(SOEN_PYTHON), "--version"], capture_output=True, text=True)
print(f"SOEN Python:      {SOEN_PYTHON} ({_ver.stdout.strip()})")
print(f"Kernel Python:    {sys.executable} (Python {sys.version.split()[0]})")

# ── Helper: run code in the soen-compatible Python ────────────────
def run_soen_python(code: str, check: bool = True) -> subprocess.CompletedProcess:
    """Run Python code using the soen-compatible interpreter."""
    env = {**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)}
    return subprocess.run(
        [str(SOEN_PYTHON), "-c", code],
        env=env, check=check, capture_output=True, text=True,
    )

# ── Paths ─────────────────────────────────────────────────────────
PCA8_H5 = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/pca8_100x_flattop/all_data.h5")

SOEN_DIR = Path("soen_training")
DATASET_DIR = SOEN_DIR / "datasets"
MODEL_DIR = SOEN_DIR / "model_specs"
CONFIG_DIR = SOEN_DIR / "training_configs"
RESULTS_DIR = SOEN_DIR / "results"

for d in [DATASET_DIR, MODEL_DIR, CONFIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PCA8 source:      {PCA8_H5}")
print(f"SOEN dir:         {SOEN_DIR.resolve()}")

## 1. Convert PCA8 H5 → SOEN HDF5 format

Reshape `(N, 7812, 8)` → `(N, 279, 28×8)` = `(N, 279, 224)`.
Each of 279 timesteps feeds 28 groups of 8 PCA features to the 28 neurons.

- `data`: `(N, 279, 224)` float32
- `labels`: `(N,)` int64 — one class per sequence
- `input_mask`: `(N, 279)` bool

In [ ]:
SOEN_H5 = DATASET_DIR / "pca8_disruption_classification_2class.h5"
N_NEURONS = 28   # SOEN hidden neurons
N_INPUTS = 8     # synaptic inputs per neuron (PCA components)
INPUT_DIM = N_NEURONS * N_INPUTS  # 224

with h5py.File(PCA8_H5, "r") as src, h5py.File(SOEN_H5, "w") as dst:
    for split in ("train", "val", "test"):
        g = dst.create_group(split)

        # X: (N, 8, 7812) → transpose → (N, 7812, 8) → reshape → (N, 279, 28*8)
        X = np.asarray(src[f"{split}/X"])              # (N, 8, 7812)
        X = np.transpose(X, (0, 2, 1))                 # (N, 7812, 8)
        N, T, D = X.shape
        T_steps = T // N_NEURONS                        # 279
        X = X[:, :T_steps * N_NEURONS, :]               # trim to exact multiple
        # Reshape: each step = 28 consecutive timepoints × 8 channels = 224 inputs
        X = X.reshape(N, T_steps, N_NEURONS * D)        # (N, 279, 224)
        g.create_dataset("data", data=X, dtype=np.float32)

        # labels: per-sequence class
        seq_labels = np.asarray(src[f"{split}/labels"])  # (N,) int64
        g.create_dataset("labels", data=seq_labels, dtype=np.int64)

        # input_mask: all timesteps valid
        input_mask = np.ones((N, T_steps), dtype=bool)
        g.create_dataset("input_mask", data=input_mask)

        n_pos = int((seq_labels == 1).sum())
        print(f"  {split}: N={N}, T={T}→{T_steps} steps, D={D}→{N_NEURONS*D} "
              f"({N_NEURONS}×{D}), disruptive={n_pos}/{N}")

print(f"\nSaved: {SOEN_H5}")
print(f"  Shape per sample: ({T_steps}, {INPUT_DIM}) = ({T_steps} steps × {N_NEURONS} neurons × {N_INPUTS} inputs)")
print(f"  total_time_ns: {T_steps * 10} ns")

## 2. Build SOEN model (SQUID readout + off-chip linear)

`Linear(224) → SingleDendrite(28, recurrent) → SQUID DendriteReadout(24) → Linear(2)`

- **J_0_to_1** (224→28): trainable, QAT 3-bit, [-0.14, 0.14]
- **J_1_to_1** (28→28 recurrent): trainable, QAT 3-bit, [-0.14, 0.14]
- **J_1_to_2** (first 24 of 28 → 24 SQUID): **fixed** one-to-one, J=0.5
- **J_2_to_3** (24→2): trainable, **off-chip** float (NOT quantized)

In [ ]:
MODEL_PATH = MODEL_DIR / "224IN_28H_24SQUID_2OffchipLinear.soen"

_build_code = f"""
import sys
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from notebook_utils import build_squid24_offchip_linear_model
from pathlib import Path

build_squid24_offchip_linear_model(
    model_path=Path({str(MODEL_PATH.resolve())!r}),
    output_dim=2,           # binary classification (softmax)
    input_dim=224,          # 28 neurons × 8 inputs each
    hidden_dim=28,          # 28 dendrite neurons
    readout_dim=24,         # first 24 neurons → SQUID readout
    dt_ns=10.0,
    phi_offset=0.23,
    phi_offset_learnable=False,
    bias_current=1.7,
    bias_current_learnable=False,
    gamma_plus=2.3508e-5,
    gamma_minus=2.6995e-5,
    gamma_minus_learnable=False,
)
print("OK")
"""
_r = run_soen_python(_build_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Model build failed")
print(f"Model saved: {MODEL_PATH}")
print(f"  Layer 0: Linear(224)           — input (28 neurons × 8 synapses)")
print(f"  Layer 1: SingleDendrite(28)    — hidden, recurrent")
print(f"  Layer 2: DendriteReadout(24)   — SQUID, first 24 of 28 neurons")
print(f"  Layer 3: Linear(2)             — off-chip projection → softmax")
print(f"  File exists: {MODEL_PATH.exists()}")

## 3. Build training config via `build_training_config_from_knobs()`

- `task_type='seq2static_classification'` → `mapping: seq2static`, `time_pooling: final`
- `readout_variant='offchip_linear'` → QAT only on J_0_to_1 and J_1_to_1 (not J_2_to_3)
- `num_classes: 2`

In [ ]:
# ── Run settings (same knobs as tutorial 02_train_models.ipynb) ──
EXPERIMENT_NAME = "pca8_disruption_classification_2class"
MAX_EPOCHS = 200
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
BACKEND = "jax"
DT_NS = 10.0
READOUT_VARIANT = "offchip_linear"  # SQUID(24) → Linear(2)
NUM_CLASSES = 2

USE_MINMAX_INPUT_SCALING = True
INPUT_SCALE_MIN = 0.0
INPUT_SCALE_MAX = 1.0
CACHE_STRATEGY = "lru"
CACHE_MEMORY_BUDGET_MB = 512.0

CONFIG_PATH = CONFIG_DIR / f"training_config_{EXPERIMENT_NAME}.yaml"
BASE_CONFIG_PATH = TUTORIAL_DIR / "training" / "training_configs" / "bnl_tutorial_base.yaml"

# Build config via build_training_config_from_knobs (same as tutorial)
# Auto-detects seq_len=279 from H5, sets total_time_ns=2790
_config_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from pathlib import Path
from notebook_utils import build_training_config_from_knobs

profile = build_training_config_from_knobs(
    base_config_path=Path({str(BASE_CONFIG_PATH.resolve())!r}),
    out_config_path=Path({str(CONFIG_PATH.resolve())!r}),
    dataset_path=Path({str(SOEN_H5.resolve())!r}),
    model_path=Path({str(MODEL_PATH.resolve())!r}),
    task_type='seq2static_classification',
    experiment_name={EXPERIMENT_NAME!r},
    max_epochs={MAX_EPOCHS},
    batch_size={BATCH_SIZE},
    learning_rate={LEARNING_RATE},
    backend={BACKEND!r},
    num_classes={NUM_CLASSES},
    dt_ns={DT_NS},
    readout_variant={READOUT_VARIANT!r},
    use_minmax_input_scaling={USE_MINMAX_INPUT_SCALING},
    input_scale_min={INPUT_SCALE_MIN},
    input_scale_max={INPUT_SCALE_MAX},
    cache_strategy={CACHE_STRATEGY!r},
    cache_memory_budget_mb={CACHE_MEMORY_BUDGET_MB},
)
print(json.dumps(profile, default=str))
"""

_r = run_soen_python(_config_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Config build failed")

import json as _json
_profile = _json.loads(_r.stdout.strip().split("\n")[-1])
print(f"Config saved: {CONFIG_PATH}")
print(f"  task: seq2static_classification, classes: {NUM_CLASSES}")
print(f"  readout: {READOUT_VARIANT} (QAT only on J_0_to_1, J_1_to_1)")
print(f"  epochs: {MAX_EPOCHS}, batch: {BATCH_SIZE}, lr: {LEARNING_RATE}")
print(f"  Dataset profile: {_profile}")

# Verify
_cfg_check = yaml.safe_load(CONFIG_PATH.read_text())
print(f"\n  Config verification:")
print(f"    total_time_ns:   {_cfg_check['data'].get('total_time_ns')}")
print(f"    sequence_length: {_cfg_check['data'].get('sequence_length')}")
print(f"    mapping:         {_cfg_check['training'].get('mapping')}")
print(f"    time_pooling:    {_cfg_check['model'].get('time_pooling')}")

## 4. Train

Invokes `soen_toolkit.training` via subprocess (same as tutorial `02_train_models.ipynb`).
Saves `initial.soen` and `last.soen` checkpoints in `.soen` format.

In [ ]:
# Training via subprocess using the soen-compatible Python
print(f"Launching SOEN training: {CONFIG_PATH}")
print(f"  Python: {SOEN_PYTHON}")
print(f"  This may take a while for {MAX_EPOCHS} epochs...")

_train_result = subprocess.run(
    [str(SOEN_PYTHON), "-m", "soen_toolkit.training", str(CONFIG_PATH.resolve())],
    env={**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)},
)
if _train_result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {_train_result.returncode}")
print("Training complete.")

## 5. Plot loss curves

In [ ]:
import matplotlib.pyplot as plt
import json as _json

# Read TensorBoard scalars via subprocess
_tb_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from notebook_utils import read_training_scalars_with_fallback
scalars = read_training_scalars_with_fallback({str(CONFIG_PATH.resolve())!r})
out = {{}}
for tag, df in scalars.items():
    out[tag] = {{"step": df["step"].tolist(), "value": df["value"].tolist()}}
print(json.dumps(out))
"""
_r = run_soen_python(_tb_code, check=False)
scalars = {}
if _r.returncode == 0 and _r.stdout.strip():
    try:
        raw = _json.loads(_r.stdout.strip().split("\n")[-1])
        scalars = {k: v for k, v in raw.items()}
    except _json.JSONDecodeError:
        print("Warning: could not parse TensorBoard output")
else:
    print(f"Warning: TensorBoard read failed. stderr: {_r.stderr[:500] if _r.stderr else 'none'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for tag, data in scalars.items():
    if "loss" in tag.lower() and "epoch" in tag.lower():
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(data["step"], data["value"], label=label)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
found_acc = False
for tag, data in scalars.items():
    if "acc" in tag.lower() and "epoch" in tag.lower():
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(data["step"], data["value"], label=label)
        found_acc = True
if found_acc:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Per-timestep Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No accuracy metrics logged", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

## 6. Audit trained weights and quantize to 3-bit `.soen`

Verify:
1. All trainable connections (J_0_to_1, J_1_to_1) are within [-0.14, 0.14]
2. Fixed parameters (phi_offset, bias_current, gamma, J_1_to_2) are unchanged
3. QAT produced valid 3-bit (9-level) quantized weights

**Output**: `last_quant_3bit9lvl.soen` — the hardware deliverable.

In [ ]:
import json as _json

# Audit and quantize via subprocess
_audit_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from pathlib import Path
from notebook_utils import audit_and_quantize_latest_run

audit = audit_and_quantize_latest_run(
    results_dir=Path({str(RESULTS_DIR.resolve())!r}),
    config_path=Path({str(CONFIG_PATH.resolve())!r}),
    experiment_name={EXPERIMENT_NAME!r},
    target_connections=["J_0_to_1", "J_1_to_1"],
    weight_min=-0.14,
    weight_max=0.14,
    quant_levels=9,
)

# Serialize for transfer back to notebook
out = {{
    "bounds_ok": audit["bounds_ok"],
    "fixed_params_unchanged": audit["fixed_params_unchanged"],
    "qat_active_in_config": audit["qat_active_in_config"],
    "bounds_stats": {{k: {{kk: float(vv) if isinstance(vv, (int, float)) else vv for kk, vv in v.items()}} for k, v in audit["bounds_stats"].items()}},
    "quant_levels_present": {{k: int(v) for k, v in audit.get("quant_levels_present", {{}}).items()}},
    "checkpoint_dir": str(audit["checkpoint_dir"]),
    "quantized_checkpoint": str(audit["quantized_checkpoint"]),
}}
loss_info = audit.get("loss_info", {{}})
out["loss_info"] = {{k: (float(v) if isinstance(v, (int, float)) else str(v)) for k, v in loss_info.items()}}
print(json.dumps(out))
"""

_r = run_soen_python(_audit_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Audit failed")

audit = _json.loads(_r.stdout.strip().split("\n")[-1])

print("=== Audit Results ===")
print(f"  Bounds OK:              {audit['bounds_ok']}")
print(f"  Fixed params unchanged: {audit['fixed_params_unchanged']}")
print(f"  QAT active in config:   {audit['qat_active_in_config']}")

print("\n=== Weight Bounds ===")
for conn, stats in audit["bounds_stats"].items():
    print(f"  {conn}: min={stats['min']:.6f}, max={stats['max']:.6f}, in_bounds={stats['in_bounds']}")

print("\n=== Quantization Levels ===")
for conn, n_lvl in audit.get("quant_levels_present", {}).items():
    print(f"  {conn}: {n_lvl} unique quantized values")

print("\n=== Loss (float vs quantized) ===")
loss_info = audit.get("loss_info", {})
print(f"  Train loss (float32):   {loss_info.get('train_loss_float', 'N/A')}")
print(f"  Train loss (quantized): {loss_info.get('train_loss_quantized', 'N/A')}")

print(f"\n=== Hardware Deliverable ===")
print(f"  {audit['quantized_checkpoint']}")

## 7. Verify checkpoint structure

Load the quantized `.soen` file and inspect its contents to confirm it matches the expected format for hardware deployment.

In [ ]:
# Inspect the quantized checkpoint via subprocess
_inspect_code = f"""
import sys, json, torch
sys.path.insert(0, {str(SOEN_SRC)!r})

quant_path = {str(Path(audit['quantized_checkpoint']))!r}
obj = torch.load(quant_path, map_location="cpu", weights_only=False)

out = {{"keys": list(obj.keys()), "model_type": obj.get("model_type", "N/A"), "dt_ns": obj.get("dt_ns", "N/A")}}

sd = obj.get("state_dict", {{}})
sd_info = {{}}
for k, v in sd.items():
    if hasattr(v, "shape"):
        sd_info[k] = {{"shape": list(v.shape), "dtype": str(v.dtype)}}
    else:
        sd_info[k] = {{"value": str(v)}}
out["state_dict"] = sd_info

layers = []
for lc in obj.get("layers_config", []):
    layers.append({{"id": lc.get("id"), "type": lc.get("type"), "dim": lc.get("dim"), "desc": lc.get("description", "")}})
out["layers"] = layers

conns = []
for cc in obj.get("connections_config", []):
    conns.append({{"src": cc.get("source_layer_id"), "tgt": cc.get("target_layer_id"),
                  "structure": cc.get("structure", {{}}).get("type", "?"), "learnable": cc.get("learnable")}})
out["connections"] = conns

print(json.dumps(out))
"""

_r = run_soen_python(_inspect_code, check=False)
if _r.returncode == 0:
    info = _json.loads(_r.stdout.strip().split("\n")[-1])

    print(f"=== .soen checkpoint ===")
    print(f"Top-level keys: {info['keys']}")
    print(f"model_type: {info['model_type']}")
    print(f"dt_ns:      {info['dt_ns']}")

    print(f"\n=== state_dict ===")
    for k, v in info["state_dict"].items():
        if "shape" in v:
            print(f"  {k}: shape={v['shape']}, dtype={v['dtype']}")
        else:
            print(f"  {k}: {v['value']}")

    print(f"\n=== Layers ===")
    for l in info["layers"]:
        print(f"  Layer {l['id']}: {l['type']} dim={l['dim']} — {l['desc']}")

    print(f"\n=== Connections ===")
    for c in info["connections"]:
        print(f"  {c['src']}→{c['tgt']}: {c['structure']}, learnable={c['learnable']}")
else:
    print("Checkpoint inspection failed:")
    print(_r.stderr)